In [ ]:
# =================================================================
# STEP 0 & 1 — Fusion progressive + Diagnostics par morceaux
# =================================================================
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import gc

# 1. Configuration des chemins
ROOT    = Path("C:/Users/Théo Lassale/Desktop/Perso/Stage/Stage_Malmö/IMU_LM_Data")
CLEANED = ROOT / "data" / "cleaned_premerge"
MERGED  = ROOT / "data" / "merged_dataset"
MERGED.mkdir(parents=True, exist_ok=True)

out_path = MERGED / "unified_dataset.parquet"

DATASETS = {
    "opportunity": "opportunity_clean_data.parquet",
    "pamap2":      "pamap2_clean_data.parquet",
    "recofit":     "recofit_clean_data.parquet",
    "ut_watch":    "ut_watch_clean_data.parquet",
    "wear":        "wear_clean_data.parquet",
    "wisdm":       "wisdm_clean_data.parquet",
    "samosa":      "samosa_clean_data.parquet",
    "capture24":   "capture24_clean_data.parquet"
}

COLS = [
    "dataset", "subject_id", "session_id", "timestamp_ns",
    "acc_x", "acc_y", "acc_z",
    "global_activity_id", "global_activity_label",
    "dataset_activity_id", "dataset_activity_label"
]

TARGET_SCHEMA = pa.schema([
    ("dataset", pa.string()), ("subject_id", pa.string()), ("session_id", pa.string()),
    ("timestamp_ns", pa.int64()), ("acc_x", pa.float32()), ("acc_y", pa.float32()), ("acc_z", pa.float32()),
    ("global_activity_id", pa.int32()), ("global_activity_label", pa.string()),
    ("dataset_activity_id", pa.int32()), ("dataset_activity_label", pa.string())
])

# --- PHASE 1 : FUSION PROGRESSIVE ---
writer = None
total_rows = 0

print(f"🚀 Début de la fusion progressive vers : {out_path}")

for ds_name, fname in DATASETS.items():
    p = CLEANED / fname
    if not p.exists(): continue

    print(f"[PROCESS] {ds_name:12s}...", end=" ", flush=True)
    pf = pq.ParquetFile(p)
    
    for batch in pf.iter_batches(batch_size=1_000_000, columns=COLS):
        table = pa.Table.from_batches([batch])
        cast_cols = [table.column(f.name).cast(f.type) for f in TARGET_SCHEMA]
        final_batch = pa.Table.from_arrays(cast_cols, schema=TARGET_SCHEMA)
        
        if writer is None:
            writer = pq.ParquetWriter(out_path, TARGET_SCHEMA, compression="zstd")
        
        writer.write_table(final_batch)
        total_rows += len(final_batch)
        del table, final_batch
        gc.collect()
    print("OK")

if writer:
    writer.close()

# --- PHASE 2 : DIAGNOSTICS PAR MORCEAUX (Anti-Crash) ---
print("\n" + "="*40)
print("📊 ANALYSE DU DATASET GÉNÉRÉ (PAR MORCEAUX)")
print("="*40)

# Initialisation des compteurs pour les stats
ds_counts = {}
sub_counts = {}
sess_counts = {}
null_counts = 0
mapping_pairs = []
label_counts = {}
UNKNOWN_ID = 9000
non_unknown_count = 0

pf_final = pq.ParquetFile(out_path)

for batch in pf_final.iter_batches(batch_size=1_000_000):
    df_chunk = batch.to_pandas()
    
    # 3) Basic stats (Rows, Subjects, Sessions)
    for ds in df_chunk["dataset"].unique():
        mask = df_chunk["dataset"] == ds
        ds_counts[ds] = ds_counts.get(ds, 0) + mask.sum()
        
        # Pour subject et session, on collecte les IDs uniques
        if ds not in sub_counts: sub_counts[ds] = set()
        if ds not in sess_counts: sess_counts[ds] = set()
        sub_counts[ds].update(df_chunk.loc[mask, "subject_id"].unique())
        sess_counts[ds].update(df_chunk.loc[mask, "session_id"].unique())
    
    # 4) Required-not-null coverage
    null_counts += df_chunk[COLS].notnull().all(axis=1).sum()
    
    # 5) Global mapping & label distribution
    non_unknown_count += (df_chunk["global_activity_id"] != UNKNOWN_ID).sum()
    for lbl, count in df_chunk["global_activity_label"].value_counts().items():
        label_counts[lbl] = label_counts.get(lbl, 0) + count
        
    # 6) Quick 1-1 check collection
    mapping_pairs.append(df_chunk[["global_activity_id", "global_activity_label"]].drop_duplicates())
    
    del df_chunk
    gc.collect()

# Finalisation et affichage des résultats
print(f"\nUNIFIED rows: {total_rows:,}")

print("\nRows per dataset:")
for ds, count in ds_counts.items(): print(f"{ds:15}: {count:,}")

print("\nSubjects per dataset:")
for ds, subs in sub_counts.items(): print(f"{ds:15}: {len(subs)}")

print("\nSessions per dataset:")
for ds, sess in sess_counts.items(): print(f"{ds:15}: {len(sess)}")

print(f"\nRows meeting required-not-null contract: {(null_counts/total_rows)*100:.2f}%")
print(f"Global activity mapping coverage: {(non_unknown_count/total_rows)*100:.1f}% (unknown={UNKNOWN_ID})")

print("\nTop-20 canonical labels (global_activity_label):")
sorted_labels = sorted(label_counts.items(), key=lambda x: x[1], reverse=True)[:20]
for lbl, count in sorted_labels: print(f"{lbl:25}: {count:,}")

# 6) Final 1-1 check
full_mapping = pd.concat(mapping_pairs).drop_duplicates()
id_to_lbl_counts = full_mapping.groupby("global_activity_id")["global_activity_label"].nunique()
print("\nGlobal id→label one-to-one:", bool((id_to_lbl_counts == 1).all()))

print(f"\n✅ Terminé. Le fichier est prêt ici : {out_path}")